# HeatUp — Notebook 2: Stability Analysis

Runs the three-gate stability pipeline (mechanical → vibrational → thermodynamic)
and produces all analysis plots.  **Edit only the Configuration cell.**

## Phonon modes

| `PHONON_MODE` | Method | Cost | When to use |
|---------------|--------|------|-------------|
| `"HA"`   | Harmonic Approximation (finite displacement at V₀) | Low | Competing phases, quick screening, low-T |
| `"QHA"`  | Quasi-Harmonic Approximation (phonons at N volumes) | Medium | Thermal expansion, moderate-T accuracy |
| `"VDOS"` | Anharmonic VDOS from AIMD (VACF, default) | High | Full anharmonicity, high-T materials |

Switch with one line in the Configuration cell:
```python
PHONON_MODE = "QHA"   # or "HA" or "VDOS"
```

Every result is saved alongside a `_manifest.json` recording the exact
configuration used (ensemble, timestep, calculator, thresholds, etc.).

## Gates
| Gate | Test | Inputs |
|------|------|--------|
| 1 Mechanical | Born–Huang criterion (all C eigenvalues > 0) | `elastic/elastic_tensor.json` |
| 2 Vibrational | Soft-mode fraction in VDOS near ω=0 | `aimd/<T>K/anharmonic_phonons/vdos.json` |
| 3 Thermodynamic | E_above_hull(T) at operating temperature | F(T) for target + all competing phases |

> **Idempotent**: re-running skips already-completed steps.

In [ ]:
import json, os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

import heatup.config as cfg
from heatup import run_stability_pipeline, run_stability_pipeline_batch
from heatup.md_pipeline import scan_database, load_analysis, print_database_summary
from heatup.manifest import manifest_summary, check_manifest_match
from heatup.anharmonic_phonons import (
    compute_anharmonic_phonons_for_sim,
    get_anharmonic_free_energy,
    _find_completed_sim_dirs,
    _load_cached_vdos,
)
from heatup.phonons import get_free_energy, run_phonons
from heatup.free_energy import build_default_assembler

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device  : {DEVICE}')
print(f'Model   : {cfg.MACE_MODEL}')

## Configuration — **edit only this cell**

All parameters that affect the physics or the results are set here and
recorded in `_manifest.json` files next to every output.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
DATABASE_ROOT  = 'database'
CANDIDATES_DIR = 'input/candidates'

# ── Calculator / model ─────────────────────────────────────────────────────
cfg.MACE_MODEL   = os.environ.get('MACE_MODEL_PATH', 'mace-mpa-0-medium')
cfg.CALC_BACKEND = 'mace-mp'   # 'mace-mp' | 'chgnet' | 'm3gnet' | 'custom'

# ── Phonon mode ─────────────────────────────────────────────────────────────
# Controls how F_vib(T) is computed for the TARGET material.
# Competing phases always use HA (fast) unless VDOS is already cached.
#
#   'HA'   — Harmonic approximation (finite displacement at V₀).
#            Requires: phonons/dos.json (from notebook 1).
#
#   'QHA'  — Quasi-harmonic approximation (phonons at multiple volumes).
#            Requires: phonopy installed.  Slower than HA.
#            Set QHA parameters below.
#
#   'VDOS' — Anharmonic VDOS from AIMD trajectory (default, recommended).
#            Requires: completed AIMD run from notebook 1.
cfg.PHONON_MODE = 'VDOS'   # 'HA' | 'QHA' | 'VDOS'

# Force-constant order (HA and QHA only; ignored for VDOS).
# 2 = standard harmonic IFCs (fast, both backends)
# 3 = 3rd-order IFCs via phono3py (very slow; phonopy backend only)
cfg.FORCE_CONSTANT_ORDER = 2

# Phonon calculation backend (HA and QHA only).
# 'ase'     — ASE Phonons class (no extra deps, 2nd-order only)
# 'phonopy' — phonopy (required for QHA and 3rd-order)
cfg.PHONON_BACKEND = 'ase'

# ── QHA parameters (only used when PHONON_MODE = 'QHA') ────────────────────
cfg.QHA_N_VOLUMES    = 7      # number of volume points (must be odd, ≥ 3)
cfg.QHA_VOLUME_RANGE = 0.06   # ±6 % volume range around V₀
cfg.QHA_EOS          = 'vinet' # EOS model: 'vinet' | 'birch_murnaghan' | 'murnaghan'

# ── MD ensemble parameters (relevant for VDOS mode, sets context for analysis) ─
# These should match what was used in notebook 1 (read from simulation-input.json).
# They are recorded in the manifest for traceability.
cfg.MD_ENSEMBLE   = 'NPT'   # 'NPT' | 'NVT' (must match notebook 1)
cfg.MD_TIMESTEP_FS = 1.0
cfg.MD_N_STEPS     = 30_000
cfg.MD_NBLOCK      = 20
cfg.MD_STEP_EQUIV  = 100
cfg.MD_TTIME_FS    = 50.0
cfg.MD_PTIME_FS    = 500.0
cfg.MD_PRESSURE_GPA = 0.0

# ── Temperature grid ───────────────────────────────────────────────────────
HULL_TEMPERATURES = list(range(0, 1501, 50))  # K
cfg.HULL_TEMPERATURES = HULL_TEMPERATURES

# Operating temperature: the specific T at which stability is evaluated
# (e.g. synthesis temperature or operating temperature of the device).
OPERATING_T = 800.0  # K

# ── Stability thresholds ────────────────────────────────────────────────────
cfg.THERMO_HULL_WARN_EV   = 0.10  # meV above hull → metastable (100 meV/atom)
cfg.VIB_ZERO_WINDOW_MEV   = 1.0   # soft-mode inspection window (meV)
cfg.VIB_ZERO_FRAC_WARN    = 0.02  # 2% → warn
cfg.VIB_ZERO_FRAC_FAIL    = 0.08  # 8% → fail

# ── Competing-phase sources for the thermodynamic hull ─────────────────────
# List controls which sources are queried, in order.
# Remove 'mp-api' to skip Materials Project download (requires MP_API_KEY).
# Remove 'pyxtal' to skip random structure generation (saves time).
cfg.COMPETING_PHASE_SOURCES = ['mp-api', 'database', 'candidates', 'pyxtal']
cfg.MP_API_KEY = os.environ.get('MP_API_KEY', '')  # set via env var

# ── Run options ────────────────────────────────────────────────────────────
RUN_MODE          = 'batch'   # 'batch' (all DB entries) or 'manual'
MANUAL_LIST       = [('LGPS','P42-nmc'), ('Li3PS4','Pmn2_1'),
                     ('AgI','P6_3mc'),   ('Li2O','Fm-3m')]
GENERATE_MISSING  = True    # run PyXtal + MP-API for competing phases
FORCE_RERUN       = False   # set True to recompute even if cached
SAVE_FIGURES      = True

# ── Free-energy contributions ───────────────────────────────────────────────
# Controls which terms are summed into G(T) = E₀ + F_vib + F_el + ...
# Set to False to exclude a contribution (e.g. if data files are absent).
INCLUDE_ELECTRONIC      = True   # needs electronic/edos.json
INCLUDE_MAGNETIC        = True   # needs magnetic/moments.json
INCLUDE_CONFIGURATIONAL = True   # needs disorder/site_occupancies.json
INCLUDE_PV              = False  # needs equation_of_state/eos.json

os.makedirs(DATABASE_ROOT, exist_ok=True)

print(f'PHONON_MODE      : {cfg.PHONON_MODE}')
print(f'IFC order        : {cfg.FORCE_CONSTANT_ORDER}')
print(f'Phonon backend   : {cfg.PHONON_BACKEND}')
print(f'Calculator       : {cfg.CALC_BACKEND}:{cfg.MACE_MODEL}')
print(f'MD ensemble      : {cfg.MD_ENSEMBLE}')
print(f'Operating T      : {OPERATING_T} K')
print(f'Hull T range     : {min(HULL_TEMPERATURES)}–{max(HULL_TEMPERATURES)} K  '
      f'({len(HULL_TEMPERATURES)} points)')
print(f'Competing sources: {cfg.COMPETING_PHASE_SOURCES}')
print(f'Free-energy terms: vib({cfg.PHONON_MODE})')
if INCLUDE_ELECTRONIC:      print('  + F_el (electronic)')
if INCLUDE_MAGNETIC:        print('  + F_mag (magnetic)')
if INCLUDE_CONFIGURATIONAL: print('  + F_conf (configurational)')
if INCLUDE_PV:              print('  + PV (pressure-volume)')

## Prerequisite check

Shows which calculations from notebook 1 are complete for each material.
Missing files (✗) do not abort the pipeline — gates simply report `missing`.

In [ ]:
print_database_summary(DATABASE_ROOT)

print(f'\n{"Material/Symmetry":<35} {"Elastic":>7} {"DOS":>5} {"E0":>4} {"AIMD":>6} {"VDOS":>6} {"QHA":>5}')
print('-' * 68)

seen = set()
for rec in scan_database(DATABASE_ROOT):
    key = f"{rec['material']}/{rec['symmetry']}"
    if key in seen: continue
    seen.add(key)
    sd = os.path.join(DATABASE_ROOT, rec['material'], rec['symmetry'])
    e  = lambda b: '\u2713' if b else '\u2717'

    has_el   = os.path.exists(os.path.join(sd, 'elastic', 'elastic_tensor.json'))
    has_dos  = os.path.exists(os.path.join(sd, 'phonons', 'dos.json'))
    has_e0   = os.path.exists(os.path.join(sd, 'relaxation', 'energy.json'))
    has_qha  = os.path.exists(os.path.join(sd, 'phonons', 'qha', 'qha_result.json'))

    aimd = os.path.join(sd, 'aimd')
    has_aimd = os.path.isdir(aimd) and any(
        os.path.exists(os.path.join(aimd, tf, 'output.traj'))
        for tf in (os.listdir(aimd) if os.path.isdir(aimd) else [])
        if tf.endswith('K')
    )
    has_vdos = any(
        os.path.exists(os.path.join(aimd, tf, 'anharmonic_phonons', 'vdos.json'))
        for tf in (os.listdir(aimd) if os.path.isdir(aimd) else [])
        if tf.endswith('K')
    )

    print(f'{key:<35} {e(has_el):>7} {e(has_dos):>5} {e(has_e0):>4} '
          f'{e(has_aimd):>6} {e(has_vdos):>6} {e(has_qha):>5}')

## Run stability pipeline

The assembler below uses the `PHONON_MODE` set in the Configuration cell.
The exact assembler configuration is printed before running.

In [ ]:
import functools
from heatup.free_energy import build_default_assembler

# Build the G(T) assembler with the configured contributions.
# 'phonon_mode' maps: VDOS→'anharmonic', HA/QHA→'harmonic' (HA or QHA
# free energies are loaded by the pipeline via heatup.phonons.get_free_energy).
_phonon_mode_for_assembler = (
    'anharmonic' if cfg.PHONON_MODE == 'VDOS'
    else cfg.PHONON_MODE.lower()   # 'ha' or 'qha'
)

assembler = build_default_assembler(
    phonon_mode             = _phonon_mode_for_assembler,
    include_electronic      = INCLUDE_ELECTRONIC,
    include_magnetic        = INCLUDE_MAGNETIC,
    include_configurational = INCLUDE_CONFIGURATIONAL,
    include_pv              = INCLUDE_PV,
    device                  = DEVICE,
)

print(f'G(T) assembler contributions:')
print(f'  F_vib : {cfg.PHONON_MODE} (IFC order={cfg.FORCE_CONSTANT_ORDER}, '
      f'backend={cfg.PHONON_BACKEND})')
if INCLUDE_ELECTRONIC:      print('  F_el  : electronic DOS (if edos.json present)')
if INCLUDE_MAGNETIC:        print('  F_mag : magnetic moments (if moments.json present)')
if INCLUDE_CONFIGURATIONAL: print('  F_conf: site occupancies (if site_occupancies.json present)')
if INCLUDE_PV:              print('  PV    : pressure-volume (if eos.json present)')

# ── Run pipeline ────────────────────────────────────────────────────────────
all_reports = []

if RUN_MODE == 'batch':
    all_reports = run_stability_pipeline_batch(
        database_root           = DATABASE_ROOT,
        candidates_root         = CANDIDATES_DIR,
        operating_T             = OPERATING_T,
        temperatures            = HULL_TEMPERATURES,
        device                  = DEVICE,
        generate_missing_phases = GENERATE_MISSING,
        force_rerun             = FORCE_RERUN,
        save_figures            = SAVE_FIGURES,
    )
else:
    for mat, sym in MANUAL_LIST:
        sd = os.path.join(DATABASE_ROOT, mat, sym)
        if not os.path.isdir(sd): continue
        all_reports.append(run_stability_pipeline(
            sym_dir                 = sd,
            operating_T             = OPERATING_T,
            candidates_root         = CANDIDATES_DIR,
            database_root           = DATABASE_ROOT,
            temperatures            = HULL_TEMPERATURES,
            device                  = DEVICE,
            generate_missing_phases = GENERATE_MISSING,
            force_rerun             = FORCE_RERUN,
            save_figure             = SAVE_FIGURES,
        ))

n_ok   = sum(1 for r in all_reports if r['overall'] == cfg.STATUS_OK)
n_warn = sum(1 for r in all_reports if r['overall'] == cfg.STATUS_WARN)
n_fail = sum(1 for r in all_reports if r['overall'] == cfg.STATUS_FAIL)
print(f'\nAssessed: {len(all_reports)}  OK: {n_ok}  WARN: {n_warn}  FAIL: {n_fail}')

## Results table

In [ ]:
rows = []
for r in all_reports:
    m, v, t = r.get('mechanical',{}), r.get('vibrational',{}), r.get('thermodynamic',{})
    rows.append({
        'Material'   : r.get('material','?'),
        'Symmetry'   : r.get('symmetry','?'),
        'Mech'       : m.get('status','-'),
        'B (GPa)'    : f"{m['bulk_modulus_GPa']:.0f}"  if m.get('available') else '-',
        'G (GPa)'    : f"{m['shear_modulus_GPa']:.0f}" if m.get('available') else '-',
        'Born'       : '\u2713' if m.get('born_stable') else ('\u2717' if m.get('available') else '-'),
        'Vib'        : v.get('status','-'),
        'ζ (%)'      : f"{v['zero_window_frac']*100:.1f}" if v.get('available') else '-',
        'Thermo'     : t.get('status','-'),
        'ΔE_hull (meV)': f"{t['e_above_hull_at_T_eV']*1000:.1f}" if t.get('available') else '-',
        'Overall'    : r.get('overall','-'),
        'PhononMode' : cfg.PHONON_MODE,
    })

def _cell_color(v):
    return {'ok': 'background-color:#d5f5e3',
            'warn': 'background-color:#fef9e7',
            'fail': 'background-color:#fadbd8'}.get(str(v).lower(), '')

df = pd.DataFrame(rows)
display(
    df.style
      .applymap(_cell_color, subset=['Mech','Vib','Thermo','Overall'])
      .set_caption(f'Stability at T_op={OPERATING_T:.0f} K  |  '
                   f'Phonon mode: {cfg.PHONON_MODE}  |  '
                   f'Calculator: {cfg.CALC_BACKEND}:{cfg.MACE_MODEL}')
)

## Hull vs temperature

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
cols = plt.cm.tab10.colors

for i, r in enumerate(all_reports):
    t = r.get('thermodynamic', {})
    if not t.get('available'): continue
    valid = [(h['T'], h['e_above_hull_eV_atom'])
             for h in t.get('hull_results', [])
             if h.get('e_above_hull_eV_atom') is not None]
    if not valid: continue
    T_a = np.array([v[0] for v in valid])
    E_m = np.array([v[1] for v in valid]) * 1000
    ax.plot(T_a, E_m, color=cols[i % 10], lw=1.5,
            label=f"{r['material']}/{r['symmetry']}")

ax.axhline(0, color='green', lw=1.0, ls='--', label='Stable (E=0)')
ax.axhline(cfg.THERMO_HULL_WARN_EV * 1000, color='orange', lw=1.0, ls='--',
           label=f'Warn ({cfg.THERMO_HULL_WARN_EV*1000:.0f} meV)')
ax.axvline(OPERATING_T, color='gray', lw=0.9, ls=':', label=f'T_op={OPERATING_T:.0f} K')
ax.set_xlabel('Temperature (K)')
ax.set_ylabel('ΔE_hull (meV/atom)')
ax.set_title(f'T-dependent hull  |  {cfg.PHONON_MODE} phonons  |  {cfg.CALC_BACKEND}:{cfg.MACE_MODEL}')
ax.legend(fontsize=7, frameon=False)
ax.set_xlim(0, max(HULL_TEMPERATURES))
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(f'hull_vs_temperature_{cfg.PHONON_MODE}.pdf', dpi=150, bbox_inches='tight')
plt.show()

## VDOS dashboards

Shows the anharmonic VDOS for each material/temperature with the soft-mode
inspection window highlighted.  Available when `PHONON_MODE = 'VDOS'` or when
AIMD trajectories exist regardless of the active mode.

In [ ]:
mat_sym_list = sorted({(r['material'], r['symmetry']) for r in all_reports})

C = {'blue':'#0072B2','orange':'#E69F00','green':'#009E73',
     'red':'#D55E00','sky':'#56B4E9','purple':'#CC79A7'}

for mat, sym in mat_sym_list:
    sd   = os.path.join(DATABASE_ROOT, mat, sym)
    aimd = os.path.join(sd, 'aimd')
    if not os.path.isdir(aimd): continue

    sim_dirs = _find_completed_sim_dirs(aimd)
    entries  = []
    for sd_sim in sim_dirs:
        vd = _load_cached_vdos(sd_sim)
        if vd:
            entries.append((os.path.basename(sd_sim), vd))

    if not entries:
        print(f'  {mat}/{sym}: no cached VDOS — run AIMD and anharmonic analysis first.')
        continue

    n = len(entries)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3.2), squeeze=False)
    axes = axes[0]

    for ax, (tf, vd) in zip(axes, entries):
        om = np.array(vd['omega_mev'])
        g  = np.array(vd['vdos'])

        ax.fill_between(om, 0, g, alpha=0.5, color=C['blue'])
        ax.plot(om, g, color=C['blue'], lw=0.9)

        win  = cfg.VIB_ZERO_WINDOW_MEV
        ax.axvspan(0, win, color=C['red'], alpha=0.25, label=f'|ω|<{win} meV')
        mask = om <= win
        zeta = float(np.trapezoid(g[mask], om[mask])) if mask.any() else 0.0
        col  = (C['red']    if zeta >= cfg.VIB_ZERO_FRAC_FAIL else
                C['orange'] if zeta >= cfg.VIB_ZERO_FRAC_WARN else C['green'])
        ax.text(0.97, 0.96, f'ζ={zeta*100:.1f}%', transform=ax.transAxes,
                ha='right', va='top', fontsize=9, color=col, fontweight='bold')

        ax.set_xlabel('ω (meV)')
        ax.set_title(tf)
        ax.set_xlim(om.min(), om.max())
        ax.set_ylim(bottom=0)

    fig.suptitle(f'{mat}/{sym}  —  VDOS (AIMD VACF)', fontsize=10, fontweight='bold')
    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(f'vdos_{mat}_{sym}.pdf', dpi=150, bbox_inches='tight')
    plt.show()

## G(T) free-energy decomposition

Stacked-area chart showing each contribution to G(T) = E₀ + F_vib + F_el + …
The active phonon mode (`PHONON_MODE`) is used for F_vib.

In [ ]:
T_arr = np.array(HULL_TEMPERATURES, dtype=float)

contrib_colors = {
    'F_vib_anharmonic' : '#0072B2',
    'F_vib_ha'         : '#56B4E9',
    'F_vib_qha'        : '#009E73',
    'F_vib_harmonic'   : '#56B4E9',  # fallback label from old assembler
    'F_el'             : '#E69F00',
    'F_mag'            : '#CC79A7',
    'F_conf'           : '#009E73',
    'PV'               : '#D55E00',
}

for mat, sym in mat_sym_list:
    sd = os.path.join(DATABASE_ROOT, mat, sym)
    if not os.path.exists(os.path.join(sd, 'relaxation', 'energy.json')):
        continue

    result = assembler.compute(sd, T_arr)
    if result.get('E0_eV_per_atom') is None:
        print(f'  {mat}/{sym}: no E0 — skipping G(T) plot.')
        continue

    E0 = result['E0_eV_per_atom']
    G  = np.array(result['G_eV_per_atom'])

    fig, ax = plt.subplots(figsize=(7, 3.5))
    base = np.zeros_like(T_arr)
    for name, vals in result.get('contributions', {}).items():
        v   = np.array(vals) * 1000  # eV → meV
        col = contrib_colors.get(name, '#aaaaaa')
        ax.fill_between(T_arr, base, base + v, alpha=0.7, color=col,
                        label=name.replace('_', ' '))
        base += v

    ax.plot(T_arr, (G - E0) * 1000, color='black', lw=1.5, label='G_tot')
    ax.axvline(OPERATING_T, color='gray', lw=0.9, ls=':', label=f'T_op={OPERATING_T:.0f} K')
    ax.set_xlabel('T (K)')
    ax.set_ylabel('G − E₀ (meV/atom)')
    ax.set_title(f'{mat}/{sym}  |  F_vib: {cfg.PHONON_MODE}')
    ax.legend(frameon=False, fontsize=6, ncol=2)
    plt.tight_layout()
    if SAVE_FIGURES:
        plt.savefig(f'gibbs_{mat}_{sym}_{cfg.PHONON_MODE}.pdf', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  {mat}/{sym}: active contributions = {result.get("available_contributions", [])}')

## Configuration manifest

The exact configuration used for this run is printed below and has been
saved alongside every output file as `*_manifest.json`.

In [ ]:
from heatup.manifest import build_manifest
import json as _json

manifest = build_manifest(output_path='stability_results.json', step='analysis')
print('Run manifest:')
print(_json.dumps(manifest, indent=2))

## Export results

In [ ]:
from heatup.manifest import build_manifest

# Strip large VDOS arrays (stored in anharmonic_phonons/ sub-dirs) before saving.
slim = []
for r in all_reports:
    s = {k: v for k, v in r.items() if k != 'vibrational'}
    s['vibrational'] = {k: v for k, v in r.get('vibrational', {}).items()
                        if k not in ('omega_mev', 'vdos')}
    slim.append(s)

out = {
    'manifest' : build_manifest('stability_results.json', step='analysis'),
    'reports'  : slim,
}
with open('stability_results.json', 'w') as fh:
    json.dump(out, fh, indent=2)

print(f'Saved {len(slim)} reports → stability_results.json')
print(f'Phonon mode  : {cfg.PHONON_MODE}')
print(f'IFC order    : {cfg.FORCE_CONSTANT_ORDER}')
print(f'Calculator   : {cfg.CALC_BACKEND}:{cfg.MACE_MODEL}')
for r in slim:
    print(f"  {r['material']}/{r['symmetry']:<20}  {r.get('overall','?').upper()}")